# A* 算法与遗传算法求解华容道问题：学生练习版

本 Notebook 基于完成版 `huarongdao_astar_genetic.ipynb` 制作，用于考查学生对 A* 搜索算法和华容道状态空间建模的掌握情况。

练习要求：

1. 不直接复制完成版代码。
2. 根据函数注释、参数说明和 `TODO` 提示补全关键函数。
3. 补全后运行完整程序，观察 A* 是否能够找到华容道解路径。
4. 输出路径时只保留操作步骤，不需要打印每一步状态。


## 1. A* 搜索算法相关知识

A* 搜索算法是一种常用的启发式搜索算法，适合求解状态空间搜索问题。

它会为每个状态计算一个综合评价函数：

```text
f(n) = g(n) + h(n)
```

其中：

- `g(n)`：从初始状态走到当前状态 `n` 已经付出的代价。
- `h(n)`：从当前状态 `n` 到目标状态的估计代价，也称启发函数。
- `f(n)`：综合估计代价，A* 每次优先扩展 `f(n)` 最小的状态。

A* 的关键点：

1. 状态表示要清晰，能够唯一描述当前局面。
2. 后继状态生成要准确，只生成合法动作。
3. 启发函数要尽可能有效，引导搜索更快接近目标。
4. 如果启发函数不高估真实代价，A* 通常可以保证找到最优解。

在华容道问题中：

- 状态：所有棋子的左上角坐标。
- 动作：某个棋子向上、下、左、右移动一格。
- 目标：曹操到达底部出口。
- 代价：每移动一步代价为 `1`。


## 2. A* 算法求解华容道问题的实现步骤

实现本程序时，可以按照下面顺序完成。建议先让每个小函数单独运行正确，再运行完整 A* 搜索。

1. **定义问题模型**：确定棋盘大小、棋子尺寸、初始状态和目标状态。
2. **设计状态表示**：用每个棋子的左上角坐标组成一个 `tuple` 表示当前局面。
3. **状态归一化**：将形状相同的 4 个士兵位置排序，减少重复搜索。
4. **构造棋盘占用表**：根据状态生成二维棋盘，判断每个格子是否被占用。
5. **判断目标状态**：检查曹操是否到达出口位置。
6. **生成合法后继状态**：枚举每个棋子的上下左右移动，过滤越界和碰撞动作。
7. **设计启发函数**：用曹操到出口的距离和出口通道阻挡数量估计剩余代价。
8. **执行 A* 搜索**：用优先队列按照 `f(n)=g(n)+h(n)` 选择最值得扩展的状态。
9. **记录路径信息**：用 `parent` 字典保存每个状态的前驱状态和移动动作。
10. **恢复并输出路径**：到达目标后，从目标状态反向回溯，得到完整操作步骤。

本练习中 `normalize_state` 和 `build_board` 已经给出完整实现，不作为考查内容。学生补全代码时，重点关注 `can_place_piece`、`get_neighbors`、`heuristic` 和 `astar_search` 这些函数。


## 3. 使用 A* 算法求解华容道：学生练习程序

下面代码单元是完整练习程序。`normalize_state` 和 `build_board` 已经补齐，其余需要补全的重点函数集中在这一个程序单元格中。请根据函数注释和 `TODO` 提示完成代码。


In [ ]:
from heapq import heappop, heappush
from itertools import count

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# ============================================================
# 1. 棋盘与棋子定义
# ============================================================
# 棋盘宽度为 4，高度为 5。
BOARD_WIDTH = 4
BOARD_HEIGHT = 5

# 曹操到达该左上角坐标时，表示成功到达出口。
GOAL_CAOCAO_POSITION = (3, 1)

# 棋子定义格式：棋子名称、宽度、高度、显示字符、颜色。
# 注意：四个士兵分别命名为 Soldier1、Soldier2、Soldier3、Soldier4。
PIECES = (
    ('CaoCao', 2, 2, 'C', '#d94f45'),
    ('ZhangFei', 1, 2, 'Z', '#7b4ab2'),
    ('ZhaoYun', 1, 2, 'Y', '#3b82c4'),
    ('MaChao', 1, 2, 'M', '#2f9e44'),
    ('HuangZhong', 1, 2, 'H', '#8f6b32'),
    ('GuanYu', 2, 1, 'G', '#e0a526'),
    ('Soldier1', 1, 1, 'S1', '#8d99ae'),
    ('Soldier2', 1, 1, 'S2', '#8d99ae'),
    ('Soldier3', 1, 1, 'S3', '#8d99ae'),
    ('Soldier4', 1, 1, 'S4', '#8d99ae'),
)

# 经典横刀立马初始布局。
# 每个元组表示对应棋子左上角位置：(row, col)。
INITIAL_STATE = (
    (0, 1),  # CaoCao
    (0, 0),  # ZhangFei
    (0, 3),  # ZhaoYun
    (2, 0),  # MaChao
    (2, 3),  # HuangZhong
    (2, 1),  # GuanYu
    (3, 1),  # Soldier1
    (3, 2),  # Soldier2
    (4, 0),  # Soldier3
    (4, 3),  # Soldier4
)


# ============================================================
# 2. 状态处理函数
# ============================================================
# 函数功能：对状态进行归一化，减少搜索时的重复状态。
# 求解作用：士兵形状完全相同，排序后可以把等价局面视为同一个状态。
def normalize_state(state):
    """
    状态归一化。

    程序中 4 个士兵分别命名为 Soldier1、Soldier2、Soldier3、Soldier4，
    用于路径输出时区分显示。

    但在搜索时，四个士兵形状完全相同。为了减少等价重复状态，
    仍然将后 4 个士兵的位置排序后作为搜索状态。
    这样既能在输出中看到 Soldier1-4，又能保证 A* 搜索效率。
    """
    fixed_pieces = list(state[:6])
    soldiers = sorted(state[6:])
    return tuple(fixed_pieces + soldiers)


# 函数功能：根据每个棋子的坐标构造棋盘占用表。
# 求解作用：后续判断移动是否合法时，需要快速知道每个格子被哪个棋子占用。
def build_board(state):
    """
    根据状态构造棋盘占用表。

    返回：
        board[row][col] 为 None 或棋子编号。
    """
    board = [[None for _ in range(BOARD_WIDTH)] for _ in range(BOARD_HEIGHT)]

    for piece_index, (row, col) in enumerate(state):
        name, width, height, symbol, color = PIECES[piece_index]

        for dr in range(height):
            for dc in range(width):
                r = row + dr
                c = col + dc

                if r < 0 or r >= BOARD_HEIGHT or c < 0 or c >= BOARD_WIDTH:
                    return None
                if board[r][c] is not None:
                    return None

                board[r][c] = piece_index

    return board


# 函数功能：判断当前局面是否已经完成华容道目标。
# 求解作用：A* 每扩展一个状态都要调用该函数，确定是否可以停止搜索。
def is_goal(state):
    """
    判断当前状态是否达到目标状态。需要学生补全。

    参数：
        state: 当前华容道状态。

    返回：
        True: 曹操已经到达目标位置。
        False: 尚未到达目标位置。

    实现思路：
        曹操是第 0 个棋子，因此 state[0] 是曹操左上角位置。
        只要 state[0] == GOAL_CAOCAO_POSITION，就说明求解成功。
    """
    # TODO 5: 判断曹操位置是否等于目标位置。
    pass


# ============================================================
# 3. 合法动作生成
# ============================================================
# 函数功能：判断指定棋子移动到新坐标后是否仍然合法。
# 求解作用：过滤越界或发生碰撞的非法动作，只保留可用于搜索的动作。
def can_place_piece(board, piece_index, new_row, new_col):
    """
    判断某个棋子能否放置到新位置。需要学生补全。

    参数：
        board: 当前棋盘占用表。
        piece_index: 要移动的棋子编号。
        new_row: 移动后棋子左上角行号。
        new_col: 移动后棋子左上角列号。

    返回：
        True: 可以放置。
        False: 不能放置。

    合法条件：
        1. 棋子不能越过棋盘边界。
        2. 棋子移动后不能与其他棋子重叠。
        3. 检查碰撞时，棋子原来占据的位置属于自己，可以忽略。

    实现提示：
        occupied_by = board[new_row + dr][new_col + dc]
        如果 occupied_by 不是 None 且 occupied_by != piece_index，说明发生碰撞。
    """
    # TODO 6: 读取棋子的宽度和高度。
    # name, width, height, symbol, color = PIECES[piece_index]

    # TODO 7: 判断是否越界。

    # TODO 8: 遍历棋子移动后占据的每一个格子，判断是否与其他棋子碰撞。

    # TODO 9: 如果没有越界或碰撞，则返回 True。
    pass


# 函数功能：生成当前状态一步移动后能够到达的所有合法后继状态。
# 求解作用：A* 通过不断扩展后继状态，在状态空间中寻找通向目标的路径。
def get_neighbors(state):
    """
    生成当前状态的所有合法后继状态。需要学生补全。

    参数：
        state: 当前华容道状态。

    返回：
        使用 yield 逐个返回 (next_state, move_description)。

    其中：
        next_state: 移动一步后的新状态。
        move_description: 移动说明，例如 'CaoCao Down'。

    实现过程：
        1. 调用 build_board(state) 得到当前棋盘占用表。
        2. 定义四个方向：Up、Down、Left、Right。
        3. 遍历每个棋子。
        4. 尝试让该棋子向每个方向移动一格。
        5. 使用 can_place_piece() 判断移动是否合法。
        6. 如果合法，生成新状态。
        7. 对新状态调用 normalize_state()。
        8. yield 新状态和动作说明。
    """
    # TODO 10: 构造当前棋盘。
    # board = ...

    # TODO 11: 定义四个移动方向。
    # directions = ...

    # TODO 12: 遍历所有棋子和所有方向，生成合法后继状态。
    # yield next_state, move_description
    pass


# ============================================================
# 4. 启发函数
# ============================================================
# 函数功能：计算当前状态到目标状态的估计代价 h(n)。
# 求解作用：启发函数用于引导 A* 优先搜索更接近出口的局面。
def heuristic(state):
    """
    A* 启发函数。需要学生补全。

    参数：
        state: 当前华容道状态。

    返回：
        h: 当前状态到目标状态的估计代价。

    本实验使用一个简单启发函数：
        h = 曹操到目标位置的曼哈顿距离 + 曹操下方通道阻挡数量

    解释：
        1. 曹操距离出口越远，代价越大。
        2. 曹操下方出口通道中如果有其他棋子阻挡，也会增加代价。

    实现步骤：
        1. 取出曹操当前位置 state[0]。
        2. 计算它到 GOAL_CAOCAO_POSITION 的曼哈顿距离。
        3. 调用 build_board(state) 得到棋盘。
        4. 检查曹操下方中间两列是否有其他棋子阻挡。
        5. 返回 distance + blocking_count。
    """
    # TODO 13: 计算曹操到目标位置的曼哈顿距离。

    # TODO 14: 统计曹操下方通道中的阻挡棋子数量。

    # TODO 15: 返回启发函数值。
    pass


# ============================================================
# 5. A* 搜索主函数
# ============================================================
# 函数功能：使用 A* 搜索算法从初始状态寻找华容道解路径。
# 求解作用：综合实际代价 g(n) 和启发代价 h(n)，用优先队列选择最有希望的状态。
def astar_search(start_state, max_expanded=600000):
    """
    使用 A* 搜索求解华容道。需要学生补全。

    参数：
        start_state: 初始状态。
        max_expanded: 最大扩展状态数量，用来防止搜索时间过长。

    返回：
        path: 解路径。每个元素为 (state, move)。
              state 是某一步状态，move 是到达该状态的动作。
        expanded_count: 搜索过程中实际扩展的状态数量。

    A* 搜索过程：
        1. 将初始状态归一化。
        2. 创建优先队列 priority_queue。
        3. 使用 g_score 记录从初始状态到每个状态的最小代价。
        4. 使用 parent 记录每个状态的前驱状态和动作，便于最后恢复路径。
        5. 每次从优先队列中取出 f = g + h 最小的状态。
        6. 如果该状态是目标状态，则根据 parent 反向恢复完整路径。
        7. 否则，遍历它的所有合法后继状态。
        8. 如果发现更短路径，则更新 g_score、parent，并把新状态加入优先队列。

    提示：
        heappush(priority_queue, (new_f, new_g, next(tie_breaker), next_state))
    """
    # A* 主流程补全步骤：
    # 1. 先调用 normalize_state() 处理初始状态。
    # 2. 创建 priority_queue，并把初始状态按 f=g+h 的优先级压入队列。
    # 3. 用 g_score 保存到达每个状态的最短步数。
    # 4. 用 parent 保存每个状态的前驱和动作，用于最终恢复路径。
    # 5. 循环弹出 f 值最小的状态，如果是目标状态就回溯路径。
    # 6. 对每个后继状态计算 new_g 和 new_f，发现更优路径时更新记录。

    # TODO 16: 初始化 start_state、priority_queue、g_score、parent、closed_set。

    # TODO 17: 将初始状态加入优先队列。

    # TODO 18: 编写 while 循环，不断弹出 f 值最小的状态。

    # TODO 19: 判断是否到达目标状态，如果到达，则恢复路径并返回。

    # TODO 20: 遍历后继状态，更新 g_score、parent 和 priority_queue。

    # TODO 21: 如果超过限制仍未找到解，返回 None 和 expanded_count。
    pass


# ============================================================
# 6. 运行 A* 并打印完整操作路径
# ============================================================
# 本段代码用于运行学生补全后的 A* 搜索，并只打印文字操作路径。
# 注意：完成上面所有 TODO 后，再运行下面代码。
INITIAL_STATE = normalize_state(INITIAL_STATE)
solution_path, expanded_count = astar_search(INITIAL_STATE)

print('扩展状态数量：', expanded_count)

if solution_path is None:
    print('没有在限制范围内找到解。请检查 TODO 函数是否补全正确。')
else:
    print('找到解。')
    print('解路径步数：', len(solution_path) - 1)
    print()
    print('完整操作路径如下：')

    for step_index, (state, move) in enumerate(solution_path):
        if step_index == 0:
            print(f'{step_index:03d}. 初始状态')
        else:
            print(f'{step_index:03d}. {move}')


# ============================================================
# 7. 可选：解路径可视化
# ============================================================
# 函数功能：把某个华容道状态绘制成颜色块棋盘。
# 求解作用：该函数用于观察搜索结果，不参与 A* 的状态扩展和路径计算。
def draw_state(state, title='Huarong Dao State'):
    """
    使用颜色块绘制华容道状态。

    参数：
        state: 要绘制的华容道状态。
        title: 图像标题。

    说明：
        该函数不是 A* 搜索的核心函数，主要用于观察初始状态和目标状态。
        只有当 A* 求解成功后，才能绘制 solution_path[-1][0]。
    """
    fig, ax = plt.subplots(figsize=(4.8, 6.0))
    ax.set_xlim(0, BOARD_WIDTH)
    ax.set_ylim(0, BOARD_HEIGHT)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_facecolor('#f7f2e8')
    ax.set_title(title)

    for x in range(BOARD_WIDTH + 1):
        ax.plot([x, x], [0, BOARD_HEIGHT], color='#5c4a36', linewidth=1)
    for y in range(BOARD_HEIGHT + 1):
        ax.plot([0, BOARD_WIDTH], [y, y], color='#5c4a36', linewidth=1)

    ax.plot([1, 3], [5, 5], color='#b00020', linewidth=5, solid_capstyle='round')
    ax.text(2, 4.82, 'EXIT', ha='center', va='center', color='#b00020', fontsize=10, weight='bold')

    for piece_index, (row, col) in enumerate(state):
        name, width, height, symbol, color = PIECES[piece_index]
        rect = Rectangle((col, row), width, height, facecolor=color, edgecolor='black', linewidth=1.8, alpha=0.92)
        ax.add_patch(rect)
        ax.text(col + width / 2, row + height / 2, symbol, ha='center', va='center', fontsize=12, color='white', weight='bold')

    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.show()


# 完成 TODO 并成功求解后，可以取消下面代码注释查看图像。
# draw_state(INITIAL_STATE, 'Initial State')
# if solution_path is not None:
#     draw_state(solution_path[-1][0], 'Goal State')
